# T-Lab RecSys 2026: Causal GPT-2 Sequential Recommender System
### End-to-End Training on Kaggle with NVIDIA GPU (T4 / P100) & Mixed Precision (FP16)

This notebook trains a **Causal GPT-2 Transformer from scratch** for Next-Item Prediction on the **Yambda** dataset (Yandex Music).

**Pre-flight Checklist on Kaggle:**
1. In the right sidebar under **Settings -> Accelerator**, select **GPU T4 x 2** or **GPU P100**.
2. In **Settings -> Internet**, toggle **Internet ON**.
3. Attach your Yambda dataset under `/kaggle/input/` (via "+ Add Input" in the top right), or run as-is for synthetic demo mode.

## 1. Verify GPU Accelerator

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 2. Workspace & Codebase Setup (Self-Contained)
This cell ensures that `train_base.py`, `data_utils.py`, `dataset.py`, and `model.py` are present in the active directory and registered in Python's `sys.path`. It works out of the box whether you cloned via Git or uploaded this notebook directly!

In [ ]:
import os
import sys
import base64

# 1. Check if user cloned the repo into a subfolder (using %cd magic)
subdirs = [
    "/kaggle/working/music-recommender-system",
    "/kaggle/working/Music-Recommender-System",
]
for sd in subdirs:
    if os.path.exists(os.path.join(sd, "train_base.py")):
        print(f"Found cloned repo at {sd}, switching directory...")
        %cd {sd}
        break

# 2. Self-Contained Deployment: Always sync latest modules into active working directory
EMBEDDED_MODULES = {
    "data_utils.py": "...",
    "dataset.py": "IiIiCmRhdGFzZXQucHkKLS0tLS0tLS0tLQpQeVRvcmNoIERhdGFzZXQgYW5kIERhdGFMb2FkZXIgZm9yIEF1dG9yZWdyZXNzaXZlIFNlcXVlbnRpYWwgUmVjb21tZW5kZXIgU3lzdGVtcyAoTmV4dCBJdGVtIFByZWRpY3Rpb24pLgoKSW1wbGVtZW50czoKMS4gQ2hyb25vbG9naWNhbCBhZ2dyZWdhdGlvbiBwZXIgdXNlciAoc29ydGVkIGJ5IHRpbWVzdGFtcCkuCjIuIEZpeGVkLWxlbmd0aCBzZXF1ZW5jZSB3aW5kb3dpbmcgKG1heF9sZW5ndGgpLgozLiBDYXVzYWwgbGFuZ3VhZ2UgbW9kZWxpbmcgdGFyZ2V0IGFsaWdubWVudDoKICAgLSBJbnB1dCBTZXF1ZW5jZTogIFtzXzEsIHNfMiwgLi4uLCBzX2tdCiAgIC0gU2hpZnRlZCBMYWJlbDogICBbc18yLCBzXzMsIC4uLiwgc197aysxfV0KNC4gQXR0ZW50aW9uIG1hc2tpbmcgYW5kIHByb3BlciB6ZXJvLXBhZGRpbmcgKDxQQUQ+ID0gMCkuCjUuIExpc3RlbiBkdXJhdGlvbiBhbGlnbm1lbnQgW3JfMiwgcl8zLCAuLi4sIHJfe2srMX1dIHRvIHN1cHBvcnQgUGhhc2UgMiBSTCByZXdhcmQgZmluZS10dW5pbmcuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGxvZ2dpbmcKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gdHlwaW5nIGltcG9ydCBEaWN0LCBMaXN0LCBPcHRpb25hbCwgVHVwbGUsIFVuaW9uCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaC51dGlscy5kYXRhIGltcG9ydCBEYXRhTG9hZGVyLCBEYXRhc2V0Cgpsb2dnaW5nLmJhc2ljQ29uZmlnKAogICAgbGV2ZWw9bG9nZ2luZy5JTkZPLAogICAgZm9ybWF0PSJbJShhc2N0aW1lKXNdIFslKGxldmVsbmFtZSlzXSAoJShuYW1lKXMpICUobWVzc2FnZSlzIiwKICAgIGRhdGVmbXQ9IiVZLSVtLSVkICVIOiVNOiVTIiwKKQpsb2dnZXIgPSBsb2dnaW5nLmdldExvZ2dlcigiRGF0YXNldCIpCgoKQGRhdGFjbGFzcwpjbGFzcyBSZWNCYXRjaDoKICAgICIiIgogICAgU3RydWN0dXJlZCBQeVRvcmNoIEJhdGNoIGZvciBDYXVzYWwgU2VxdWVudGlhbCBSZWNvbW1lbmRhdGlvbi4KICAgIAogICAgQXR0cmlidXRlczoKICAgICAgICBpbnB1dF9pZHM6IExvbmdUZW5zb3IgW0IsIExdIGNvbnRhaW5pbmcgaXRlbSB0b2tlbiBpbmRpY2VzLgogICAgICAgIGxhYmVsczogTG9uZ1RlbnNvciBbQiwgTF0gY29udGFpbmluZyBuZXh0LWl0ZW0gdGFyZ2V0IHRva2VuIGluZGljZXMuCiAgICAgICAgYXR0ZW50aW9uX21hc2s6IExvbmdUZW5zb3IgW0IsIExdIGNvbnRhaW5pbmcgMSBmb3IgdmFsaWQgaXRlbXMsIDAgZm9yIDxQQUQ+LgogICAgICAgIGR1cmF0aW9uczogRmxvYXRUZW5zb3IgW0IsIExdIGNvbnRhaW5pbmcgbGlzdGVuaW5nIGR1cmF0aW9uIGluIHNlY29uZHMgKFJMIHJld2FyZHMpLgogICAgICAgIHVzZXJfaWRzOiBMaXN0W1VuaW9uW3N0ciwgaW50XV0gY29udGFpbmluZyByYXcgdXNlciBpZGVudGlmaWVycyBmb3IgZXZhbHVhdGlvbiBncm91cGluZy4KICAgICIiIgogICAgaW5wdXRfaWRzOiB0b3JjaC5UZW5zb3IKICAgIGxhYmVsczogdG9yY2guVGVuc29yCiAgICBhdHRlbnRpb25fbWFzazogdG9yY2guVGVuc29yCiAgICBkdXJhdGlvbnM6IHRvcmNoLlRlbnNvcgogICAgdXNlcl9pZHM6IExpc3RbVW5pb25bc3RyLCBpbnRdXQoKICAgIGRlZiB0byhzZWxmLCBkZXZpY2U6IFVuaW9uW3RvcmNoLmRldmljZSwgc3RyXSkgLT4gUmVjQmF0Y2g6CiAgICAgICAgIiIiVHJhbnNmZXJzIGJhdGNoIHRlbnNvcnMgdG8gdGFyZ2V0IGNvbXB1dGUgZGV2aWNlIChDUFUvR1BVL01QUykuIiIiCiAgICAgICAgcmV0dXJuIFJlY0JhdGNoKAogICAgICAgICAgICBpbnB1dF9pZHM9c2VsZi5pbnB1dF9pZHMudG8oZGV2aWNlKSwKICAgICAgICAgICAgbGFiZWxzPXNlbGYubGFiZWxzLnRvKGRldmljZSksCiAgICAgICAgICAgIGF0dGVudGlvbl9tYXNrPXNlbGYuYXR0ZW50aW9uX21hc2sudG8oZGV2aWNlKSwKICAgICAgICAgICAgZHVyYXRpb25zPXNlbGYuZHVyYXRpb25zLnRvKGRldmljZSksCiAgICAgICAgICAgIHVzZXJfaWRzPXNlbGYudXNlcl9pZHMsCiAgICAgICAgKQoKCmNsYXNzIFNlcXVlbnRpYWxSZWNEYXRhc2V0KERhdGFzZXQpOgogICAgIiIiCiAgICBQeVRvcmNoIERhdGFzZXQgZ2VuZXJhdGluZyBjYXVzYWwgKGlucHV0LCBsYWJlbCkgc2VxdWVuY2UgcGFpcnMgZm9yIE5leHQgSXRlbSBQcmVkaWN0aW9uLgoKICAgIEFyZ3M6CiAgICAgICAgZGY6IERhdGFGcmFtZSBjb250YWluaW5nIGludGVyYWN0aW9uIGxvZ3MuIE11c3QgY29udGFpbiB1c2VyX2NvbCwgaXRlbV90b2tlbl9jb2wsCiAgICAgICAgICAgIHRpbWVzdGFtcF9jb2wsIGFuZCBvcHRpb25hbGx5IGR1cmF0aW9uX2NvbC4KICAgICAgICBtYXhfbGVuZ3RoOiBNYXhpbXVtIHNlcXVlbmNlIGxlbmd0aCAoY29udGV4dCB3aW5kb3cpIGZvciB0aGUgR1BULTIgbW9kZWwuCiAgICAgICAgdXNlcl9jb2w6IENvbHVtbiBuYW1lIGZvciB1c2VyIGlkZW50aWZpZXJzLgogICAgICAgIGl0ZW1fdG9rZW5fY29sOiBDb2x1bW4gbmFtZSBmb3IgbWFwcGVkIGNvbnRpbnVvdXMgaXRlbSB0b2tlbnMgKDEgLi4uIE4pLgogICAgICAgIHRpbWVzdGFtcF9jb2w6IENvbHVtbiBuYW1lIGZvciBVTklYIHRpbWVzdGFtcHMuCiAgICAgICAgZHVyYXRpb25fY29sOiBDb2x1bW4gbmFtZSBmb3IgbGlzdGVuIGR1cmF0aW9uIGluIHNlY29uZHMuCiAgICAgICAgc2xpZGluZ193aW5kb3c6IElmIFRydWUsIHVzZXMgc2xpZGluZyB3aW5kb3cgY2h1bmtpbmcgdG8gZXh0cmFjdCBtdWx0aXBsZSBzdWJzZXF1ZW5jZXMKICAgICAgICAgICAgZnJvbSBsb25nIHVzZXIgaGlzdG9yaWVzIChyZWNvbW1lbmRlZCBmb3IgdHJhaW5pbmcgZGF0YSBhdWdtZW50YXRpb24pLgogICAgICAgICAgICBJZiBGYWxzZSwgcmV0YWlucyBvbmx5IHRoZSBtb3N0IHJlY2VudCBzZXF1ZW5jZSAocmVjb21tZW5kZWQgZm9yIHZhbGlkYXRpb24vdGVzdCBldmFsdWF0aW9uKS4KICAgICAgICBzdHJpZGU6IFN0cmlkZSBzdGVwIHNpemUgd2hlbiBzbGlkaW5nX3dpbmRvdyBpcyBUcnVlLiBEZWZhdWx0cyB0byBtYXhfbGVuZ3RoIC8vIDIuCiAgICAgICAgcGFkX3Rva2VuX2lkOiBTcGVjaWFsIHRva2VuIGluZGV4IGZvciBwYWRkaW5nIChkZWZhdWx0IDApLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgZGY6IHBkLkRhdGFGcmFtZSwKICAgICAgICBoaXN0b3J5X2RmOiBPcHRpb25hbFtwZC5EYXRhRnJhbWVdID0gTm9uZSwKICAgICAgICBtYXhfbGVuZ3RoOiBpbnQgPSA2NCwKICAgICAgICB1c2VyX2NvbDogc3RyID0gInVpZCIsCiAgICAgICAgaXRlbV90b2tlbl9jb2w6IHN0ciA9ICJpdGVtX3Rva2VuIiwKICAgICAgICB0aW1lc3RhbXBfY29sOiBzdHIgPSAidGltZXN0YW1wIiwKICAgICAgICBkdXJhdGlvbl9jb2w6IHN0ciA9ICJsaXN0ZW5fZHVyYXRpb25fc2VjIiwKICAgICAgICBzbGlkaW5nX3dpbmRvdzogYm9vbCA9IEZhbHNlLAogICAgICAgIHN0cmlkZTogT3B0aW9uYWxbaW50XSA9IE5vbmUsCiAgICAgICAgcGFkX3Rva2VuX2lkOiBpbnQgPSAwLAogICAgKSAtPiBOb25lOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYubWF4X2xlbmd0aDogaW50ID0gbWF4X2xlbmd0aAogICAgICAgIHNlbGYucGFkX3Rva2VuX2lkOiBpbnQgPSBwYWRfdG9rZW5faWQKICAgICAgICBzZWxmLnNsaWRpbmdfd2luZG93OiBib29sID0gc2xpZGluZ193aW5kb3cKICAgICAgICBzZWxmLnN0cmlkZTogaW50ID0gc3RyaWRlIGlmIHN0cmlkZSBpcyBub3QgTm9uZSBlbHNlIG1heCgxLCBtYXhfbGVuZ3RoIC8vIDIpCgogICAgICAgICMgUmVzb2x2ZSB1c2VyX2NvbCBhbGlhcyBpZiBuZWVkZWQKICAgICAgICBpZiB1c2VyX2NvbCBub3QgaW4gZGYuY29sdW1uczoKICAgICAgICAgICAgaWYgInVpZCIgaW4gZGYuY29sdW1uczoKICAgICAgICAgICAgICAgIHVzZXJfY29sID0gInVpZCIKICAgICAgICAgICAgZWxpZiAidXNlcl9pZCIgaW4gZGYuY29sdW1uczoKICAgICAgICAgICAgICAgIHVzZXJfY29sID0gInVzZXJfaWQiCgogICAgICAgICMgSW4gZXZhbHVhdGlvbiAodmFsL3Rlc3QpLCBjb25jYXRlbmF0ZSBoaXN0b3JpY2FsIGludGVyYWN0aW9ucyB3aXRoIHRhcmdldCBzcGxpdAogICAgICAgICMgdG8gZW5zdXJlIHVzZXJzIGhhdmUgZnVsbCBjaHJvbm9sb2dpY2FsIGNvbnRleHQgdXAgdG8gdGhlIGV2YWx1YXRpb24gdGFyZ2V0LgogICAgICAgIGlmIGhpc3RvcnlfZGYgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICMgRmlsdGVyIGhpc3RvcnkgdG8gb25seSB1c2VycyBwcmVzZW50IGluIGN1cnJlbnQgdGFyZ2V0IHNwbGl0CiAgICAgICAgICAgIGFjdGl2ZV91c2VycyA9IHNldChkZlt1c2VyX2NvbF0udW5pcXVlKCkpCiAgICAgICAgICAgIGZpbHRlcmVkX2hpc3RvcnkgPSBoaXN0b3J5X2RmW2hpc3RvcnlfZGZbdXNlcl9jb2xdLmlzaW4oYWN0aXZlX3VzZXJzKV0KICAgICAgICAgICAgY29tYmluZWRfZGYgPSBwZC5jb25jYXQoW2ZpbHRlcmVkX2hpc3RvcnksIGRmXSwgaWdub3JlX2luZGV4PVRydWUpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgY29tYmluZWRfZGYgPSBkZgoKICAgICAgICAjIFN0b3JlIHNlcXVlbmNlIHNhbXBsZXM6IExpc3Qgb2YgdHVwbGVzICh1c2VyX2lkLCBpdGVtX3NlcSwgZHVyYXRpb25fc2VxKQogICAgICAgIHNlbGYuc2FtcGxlczogTGlzdFtUdXBsZVtVbmlvbltzdHIsIGludF0sIG5wLm5kYXJyYXksIG5wLm5kYXJyYXldXSA9IFtdCiAgICAgICAgc2VsZi5fYnVpbGRfc2VxdWVuY2VzKAogICAgICAgICAgICBkZj1jb21iaW5lZF9kZiwKICAgICAgICAgICAgdXNlcl9jb2w9dXNlcl9jb2wsCiAgICAgICAgICAgIGl0ZW1fdG9rZW5fY29sPWl0ZW1fdG9rZW5fY29sLAogICAgICAgICAgICB0aW1lc3RhbXBfY29sPXRpbWVzdGFtcF9jb2wsCiAgICAgICAgICAgIGR1cmF0aW9uX2NvbD1kdXJhdGlvbl9jb2wsCiAgICAgICAgKQoKICAgIGRlZiBfYnVpbGRfc2VxdWVuY2VzKAogICAgICAgIHNlbGYsCiAgICAgICAgZGY6IHBkLkRhdGFGcmFtZSwKICAgICAgICB1c2VyX2NvbDogc3RyLAogICAgICAgIGl0ZW1fdG9rZW5fY29sOiBzdHIsCiAgICAgICAgdGltZXN0YW1wX2NvbDogc3RyLAogICAgICAgIGR1cmF0aW9uX2NvbDogc3RyLAogICAgKSAtPiBOb25lOgogICAgICAgICIiIkdyb3VwcyB1c2VyIGludGVyYWN0aW9ucywgc29ydHMgY2hyb25vbG9naWNhbGx5LCBhbmQgY29uc3RydWN0cyBzZXF1ZW5jZXMuIiIiCiAgICAgICAgIyBFbnN1cmUgY2hyb25vbG9naWNhbCBvcmRlcmluZwogICAgICAgIHNvcnRlZF9kZiA9IGRmLnNvcnRfdmFsdWVzKGJ5PVt1c2VyX2NvbCwgdGltZXN0YW1wX2NvbF0pLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKICAgICAgICBoYXNfZHVyYXRpb24gPSBkdXJhdGlvbl9jb2wgaW4gc29ydGVkX2RmLmNvbHVtbnMKCiAgICAgICAgZ3JvdXBlZCA9IHNvcnRlZF9kZi5ncm91cGJ5KHVzZXJfY29sKQogICAgICAgIHNraXBwZWRfc2hvcnQgPSAwCgogICAgICAgIGZvciB1c2VyX2lkLCBncm91cCBpbiBncm91cGVkOgogICAgICAgICAgICBpdGVtX3NlcSA9IGdyb3VwW2l0ZW1fdG9rZW5fY29sXS52YWx1ZXMuYXN0eXBlKG5wLmludDY0KQogICAgICAgICAgICBkdXJhdGlvbl9zZXEgPSAoCiAgICAgICAgICAgICAgICBncm91cFtkdXJhdGlvbl9jb2xdLnZhbHVlcy5hc3R5cGUobnAuZmxvYXQzMikKICAgICAgICAgICAgICAgIGlmIGhhc19kdXJhdGlvbgogICAgICAgICAgICAgICAgZWxzZSBucC5vbmVzKGxlbihpdGVtX3NlcSksIGR0eXBlPW5wLmZsb2F0MzIpICogMzAuMAogICAgICAgICAgICApCgogICAgICAgICAgICAjIENhdXNhbCBMTSBuZXh0LWl0ZW0gcHJlZGljdGlvbiByZXF1aXJlcyBhdCBsZWFzdCAyIHRva2VucyAoMSBpbnB1dCwgMSBsYWJlbCkKICAgICAgICAgICAgaWYgbGVuKGl0ZW1fc2VxKSA8IDI6CiAgICAgICAgICAgICAgICBza2lwcGVkX3Nob3J0ICs9IDEKICAgICAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgICAgICBpZiBzZWxmLnNsaWRpbmdfd2luZG93OgogICAgICAgICAgICAgICAgIyBTbGlkaW5nIHdpbmRvdyBjaHVua2luZyBmb3IgdHJhaW5pbmcgZGF0YSBhdWdtZW50YXRpb24KICAgICAgICAgICAgICAgICMgQSB3aW5kb3cgb2Ygc2l6ZSAobWF4X2xlbmd0aCArIDEpIHByb3ZpZGVzIG1heF9sZW5ndGggaW5wdXRzIGFuZCBtYXhfbGVuZ3RoIHNoaWZ0ZWQgbGFiZWxzCiAgICAgICAgICAgICAgICB3aW5kb3dfc2l6ZSA9IHNlbGYubWF4X2xlbmd0aCArIDEKICAgICAgICAgICAgICAgIGlmIGxlbihpdGVtX3NlcSkgPD0gd2luZG93X3NpemU6CiAgICAgICAgICAgICAgICAgICAgc2VsZi5zYW1wbGVzLmFwcGVuZCgodXNlcl9pZCwgaXRlbV9zZXEsIGR1cmF0aW9uX3NlcSkpCiAgICAgICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgICAgIGZvciBzdGFydF9pZHggaW4gcmFuZ2UoMCwgbGVuKGl0ZW1fc2VxKSAtIDEsIHNlbGYuc3RyaWRlKToKICAgICAgICAgICAgICAgICAgICAgICAgZW5kX2lkeCA9IG1pbihzdGFydF9pZHggKyB3aW5kb3dfc2l6ZSwgbGVuKGl0ZW1fc2VxKSkKICAgICAgICAgICAgICAgICAgICAgICAgY2h1bmtfaXRlbXMgPSBpdGVtX3NlcVtzdGFydF9pZHg6ZW5kX2lkeF0KICAgICAgICAgICAgICAgICAgICAgICAgY2h1bmtfZHVyYXRpb25zID0gZHVyYXRpb25fc2VxW3N0YXJ0X2lkeDplbmRfaWR4XQogICAgICAgICAgICAgICAgICAgICAgICBpZiBsZW4oY2h1bmtfaXRlbXMpID49IDI6CiAgICAgICAgICAgICAgICAgICAgICAgICAgICBzZWxmLnNhbXBsZXMuYXBwZW5kKCh1c2VyX2lkLCBjaHVua19pdGVtcywgY2h1bmtfZHVyYXRpb25zKSkKICAgICAgICAgICAgICAgICAgICAgICAgaWYgZW5kX2lkeCA9PSBsZW4oaXRlbV9zZXEpOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgICMgRXZhbHVhdGlvbiBtb2RlOiByZXRhaW4gdGhlIG1vc3QgcmVjZW50IGhpc3RvcnkgdXAgdG8gKG1heF9sZW5ndGggKyAxKQogICAgICAgICAgICAgICAgY2h1bmtfaXRlbXMgPSBpdGVtX3NlcVstKHNlbGYubWF4X2xlbmd0aCArIDEpOl0KICAgICAgICAgICAgICAgIGNodW5rX2R1cmF0aW9ucyA9IGR1cmF0aW9uX3NlcVstKHNlbGYubWF4X2xlbmd0aCArIDEpOl0KICAgICAgICAgICAgICAgIHNlbGYuc2FtcGxlcy5hcHBlbmQoKHVzZXJfaWQsIGNodW5rX2l0ZW1zLCBjaHVua19kdXJhdGlvbnMpKQoKICAgICAgICBsb2dnZXIuaW5mbygKICAgICAgICAgICAgIkNvbnN0cnVjdGVkICVkIHNlcXVlbmNlIHNhbXBsZXMgZnJvbSAlZCB1c2VycyAoc2tpcHBlZCAlZCB1c2VycyB3aXRoIDwyIGludGVyYWN0aW9ucykuICIKICAgICAgICAgICAgInNsaWRpbmdfd2luZG93PSVzLCBtYXhfbGVuZ3RoPSVkIiwKICAgICAgICAgICAgbGVuKHNlbGYuc2FtcGxlcyksCiAgICAgICAgICAgIGxlbihncm91cGVkKSwKICAgICAgICAgICAgc2tpcHBlZF9zaG9ydCwKICAgICAgICAgICAgc2VsZi5zbGlkaW5nX3dpbmRvdywKICAgICAgICAgICAgc2VsZi5tYXhfbGVuZ3RoLAogICAgICAgICkKCiAgICBkZWYgX19sZW5fXyhzZWxmKSAtPiBpbnQ6CiAgICAgICAgcmV0dXJuIGxlbihzZWxmLnNhbXBsZXMpCgogICAgZGVmIF9fZ2V0aXRlbV9fKHNlbGYsIGlkeDogaW50KSAtPiBEaWN0W3N0ciwgVW5pb25bbnAubmRhcnJheSwgVW5pb25bc3RyLCBpbnRdXV06CiAgICAgICAgIiIiCiAgICAgICAgUmV0cmlldmVzIGEgc2luZ2xlIHNlcXVlbmNlIHNhbXBsZSwgYWxpZ25zIGlucHV0cyBhbmQgc2hpZnRlZCBjYXVzYWwgbGFiZWxzLAogICAgICAgIGFuZCBhcHBsaWVzIHplcm8tcGFkZGluZyB0byByZWFjaCBtYXhfbGVuZ3RoLgoKICAgICAgICBMZXQgcmF3IGl0ZW1zIGJlIFtzXzEsIHNfMiwgLi4uLCBzX2ssIHNfe2srMX1dLgogICAgICAgIC0gSW5wdXQgc2VxdWVuY2U6ICAgW3NfMSwgc18yLCAuLi4sIHNfa10KICAgICAgICAtIFNoaWZ0ZWQgTGFiZWw6ICAgIFtzXzIsIHNfMywgLi4uLCBzX3trKzF9XQogICAgICAgIC0gU2hpZnRlZCBEdXJhdGlvbjogW3JfMiwgcl8zLCAuLi4sIHJfe2srMX1dCiAgICAgICAgIiIiCiAgICAgICAgdXNlcl9pZCwgaXRlbXMsIGR1cmF0aW9ucyA9IHNlbGYuc2FtcGxlc1tpZHhdCgogICAgICAgICMgU2hpZnQgc2VxdWVuY2UgZm9yIG5leHQtdG9rZW4gcHJlZGljdGlvbgogICAgICAgIHJhd19pbnB1dHMgPSBpdGVtc1s6LTFdCiAgICAgICAgcmF3X2xhYmVscyA9IGl0ZW1zWzE6XQogICAgICAgIHJhd19kdXJhdGlvbnMgPSBkdXJhdGlvbnNbMTpdCgogICAgICAgICMgVHJ1bmNhdGUgaWYgbG9uZ2VyIHRoYW4gbWF4X2xlbmd0aCAoZS5nLiBmcm9tIHJpZ2h0bW9zdCByZWNlbnQgaGlzdG9yeSkKICAgICAgICBpZiBsZW4ocmF3X2lucHV0cykgPiBzZWxmLm1heF9sZW5ndGg6CiAgICAgICAgICAgIHJhd19pbnB1dHMgPSByYXdfaW5wdXRzWy1zZWxmLm1heF9sZW5ndGg6XQogICAgICAgICAgICByYXdfbGFiZWxzID0gcmF3X2xhYmVsc1stc2VsZi5tYXhfbGVuZ3RoOl0KICAgICAgICAgICAgcmF3X2R1cmF0aW9ucyA9IHJhd19kdXJhdGlvbnNbLXNlbGYubWF4X2xlbmd0aDpdCgogICAgICAgIGN1cnJfbGVuID0gbGVuKHJhd19pbnB1dHMpCiAgICAgICAgcGFkX2xlbiA9IHNlbGYubWF4X2xlbmd0aCAtIGN1cnJfbGVuCgogICAgICAgICMgUmlnaHQtcGFkZGluZyB3aXRoIHBhZF90b2tlbl9pZCAoMCkKICAgICAgICAjIElucHV0IElEczogW3Rva2VuXzEsIHRva2VuXzIsIC4uLiwgdG9rZW5faywgMCwgLi4uLCAwXQogICAgICAgIGlucHV0X2lkcyA9IG5wLnBhZChyYXdfaW5wdXRzLCAoMCwgcGFkX2xlbiksIGNvbnN0YW50X3ZhbHVlcz1zZWxmLnBhZF90b2tlbl9pZCkKICAgICAgICAjIExhYmVsczogW3Rva2VuXzIsIHRva2VuXzMsIC4uLiwgdG9rZW5fe2srMX0sIDAsIC4uLiwgMF0KICAgICAgICBsYWJlbHMgPSBucC5wYWQocmF3X2xhYmVscywgKDAsIHBhZF9sZW4pLCBjb25zdGFudF92YWx1ZXM9c2VsZi5wYWRfdG9rZW5faWQpCiAgICAgICAgIyBBdHRlbnRpb24gTWFzazogMSBmb3IgYWN0aXZlIGl0ZW0gdG9rZW5zLCAwIGZvciA8UEFEPgogICAgICAgIGF0dGVudGlvbl9tYXNrID0gbnAuemVyb3Moc2VsZi5tYXhfbGVuZ3RoLCBkdHlwZT1ucC5pbnQ2NCkKICAgICAgICBhdHRlbnRpb25fbWFza1s6Y3Vycl9sZW5dID0gMQogICAgICAgICMgRHVyYXRpb25zOiBhbGlnbmVkIHdpdGggbGFiZWxzIGZvciBSTCByZXdhcmRzCiAgICAgICAgZHVyYXRpb25zX3BhZGRlZCA9IG5wLnBhZChyYXdfZHVyYXRpb25zLCAoMCwgcGFkX2xlbiksIGNvbnN0YW50X3ZhbHVlcz0wLjApCgogICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICJpbnB1dF9pZHMiOiBpbnB1dF9pZHMsCiAgICAgICAgICAgICJsYWJlbHMiOiBsYWJlbHMsCiAgICAgICAgICAgICJhdHRlbnRpb25fbWFzayI6IGF0dGVudGlvbl9tYXNrLAogICAgICAgICAgICAiZHVyYXRpb25zIjogZHVyYXRpb25zX3BhZGRlZCwKICAgICAgICAgICAgInVzZXJfaWQiOiB1c2VyX2lkLAogICAgICAgIH0KCgpkZWYgcmVjc3lzX2NvbGxhdGVfZm4oYmF0Y2hfbGlzdDogTGlzdFtEaWN0W3N0ciwgVW5pb25bbnAubmRhcnJheSwgVW5pb25bc3RyLCBpbnRdXV1dKSAtPiBSZWNCYXRjaDoKICAgICIiIgogICAgQ29sbGF0ZSBmdW5jdGlvbiB0aGF0IGJhdGNoZXMgaW5kaXZpZHVhbCBkaWN0aW9uYXJ5IHNhbXBsZXMgaW50byBhIHVuaWZpZWQgUmVjQmF0Y2guCiAgICAiIiIKICAgIGlucHV0X2lkcyA9IHRvcmNoLmZyb21fbnVtcHkobnAuc3RhY2soW2JbImlucHV0X2lkcyJdIGZvciBiIGluIGJhdGNoX2xpc3RdKSkubG9uZygpCiAgICBsYWJlbHMgPSB0b3JjaC5mcm9tX251bXB5KG5wLnN0YWNrKFtiWyJsYWJlbHMiXSBmb3IgYiBpbiBiYXRjaF9saXN0XSkpLmxvbmcoKQogICAgYXR0ZW50aW9uX21hc2sgPSB0b3JjaC5mcm9tX251bXB5KG5wLnN0YWNrKFtiWyJhdHRlbnRpb25fbWFzayJdIGZvciBiIGluIGJhdGNoX2xpc3RdKSkubG9uZygpCiAgICBkdXJhdGlvbnMgPSB0b3JjaC5mcm9tX251bXB5KG5wLnN0YWNrKFtiWyJkdXJhdGlvbnMiXSBmb3IgYiBpbiBiYXRjaF9saXN0XSkpLmZsb2F0KCkKICAgIHVzZXJfaWRzID0gW2JbInVzZXJfaWQiXSBmb3IgYiBpbiBiYXRjaF9saXN0XQoKICAgIHJldHVybiBSZWNCYXRjaCgKICAgICAgICBpbnB1dF9pZHM9aW5wdXRfaWRzLAogICAgICAgIGxhYmVscz1sYWJlbHMsCiAgICAgICAgYXR0ZW50aW9uX21hc2s9YXR0ZW50aW9uX21hc2ssCiAgICAgICAgZHVyYXRpb25zPWR1cmF0aW9ucywKICAgICAgICB1c2VyX2lkcz11c2VyX2lkcywKICAgICkKCgpkZWYgY3JlYXRlX2RhdGFsb2FkZXIoCiAgICBkZjogcGQuRGF0YUZyYW1lLAogICAgaGlzdG9yeV9kZjogT3B0aW9uYWxbcGQuRGF0YUZyYW1lXSA9IE5vbmUsCiAgICBtYXhfbGVuZ3RoOiBpbnQgPSA2NCwKICAgIGJhdGNoX3NpemU6IGludCA9IDEyOCwKICAgIHNodWZmbGU6IGJvb2wgPSBUcnVlLAogICAgc2xpZGluZ193aW5kb3c6IGJvb2wgPSBGYWxzZSwKICAgIHN0cmlkZTogT3B0aW9uYWxbaW50XSA9IE5vbmUsCiAgICB1c2VyX2NvbDogc3RyID0gInVpZCIsCiAgICBudW1fd29ya2VyczogaW50ID0gMCwKICAgIHBpbl9tZW1vcnk6IGJvb2wgPSBGYWxzZSwKICAgIHBhZF90b2tlbl9pZDogaW50ID0gMCwKKSAtPiBEYXRhTG9hZGVyOgogICAgIiIiCiAgICBGYWN0b3J5IGZ1bmN0aW9uIGNyZWF0aW5nIGFuIG9wdGltaXplZCBQeVRvcmNoIERhdGFMb2FkZXIgZm9yIHNlcXVlbnRpYWwgcmVjb21tZW5kYXRpb24uCiAgICAiIiIKICAgIGRhdGFzZXQgPSBTZXF1ZW50aWFsUmVjRGF0YXNldCgKICAgICAgICBkZj1kZiwKICAgICAgICBoaXN0b3J5X2RmPWhpc3RvcnlfZGYsCiAgICAgICAgbWF4X2xlbmd0aD1tYXhfbGVuZ3RoLAogICAgICAgIHVzZXJfY29sPXVzZXJfY29sLAogICAgICAgIHNsaWRpbmdfd2luZG93PXNsaWRpbmdfd2luZG93LAogICAgICAgIHN0cmlkZT1zdHJpZGUsCiAgICAgICAgcGFkX3Rva2VuX2lkPXBhZF90b2tlbl9pZCwKICAgICkKCiAgICByZXR1cm4gRGF0YUxvYWRlcigKICAgICAgICBkYXRhc2V0PWRhdGFzZXQsCiAgICAgICAgYmF0Y2hfc2l6ZT1iYXRjaF9zaXplLAogICAgICAgIHNodWZmbGU9c2h1ZmZsZSwKICAgICAgICBudW1fd29ya2Vycz1udW1fd29ya2VycywKICAgICAgICBjb2xsYXRlX2ZuPXJlY3N5c19jb2xsYXRlX2ZuLAogICAgICAgIHBpbl9tZW1vcnk9cGluX21lbW9yeSwKICAgICAgICBkcm9wX2xhc3Q9RmFsc2UsCiAgICApCg==",
    "model.py": "IiIiCm1vZGVsLnB5Ci0tLS0tLS0tCkNhdXNhbCBMYW5ndWFnZSBNb2RlbCAoR1BULTIpIGZvciBBdXRvcmVncmVzc2l2ZSBOZXh0LUl0ZW0gUmVjb21tZW5kYXRpb24uCgpJbml0aWFsaXplcyBhIGNvbXBhY3QgR1BULTIgYXJjaGl0ZWN0dXJlIGZyb20gc2NyYXRjaCAobm8gcHJlLXRyYWluZWQgTkxQIHdlaWdodHMpCnRhaWxvcmVkIGZvciBpdGVtIGNhdGFsb2dzIGFuZCBzZXF1ZW50aWFsIGludGVyYWN0aW9uIGR5bmFtaWNzLgoKQXJjaGl0ZWN0dXJlIFNwZWNpZmljYXRpb25zOgogICAgLSBCYXNlIENsYXNzOiB0cmFuc2Zvcm1lcnMuR1BUMkxNSGVhZE1vZGVsIChpbml0aWFsaXplZCBmcm9tIHNjcmF0Y2gpCiAgICAtIHZvY2FiX3NpemUgPSBudW1faXRlbXMgKyAxIChJbmRleCAwIGlzIDxQQUQ+KQogICAgLSBuX2VtYmQgPSAxMjggKENvbXBhY3QgbGF0ZW50IGVtYmVkZGluZ3MpCiAgICAtIG5fbGF5ZXIgPSAzIChUcmFuc2Zvcm1lciBkZWNvZGVyIGJsb2NrcykKICAgIC0gbl9oZWFkID0gNCAoTXVsdGktaGVhZCBzZWxmLWF0dGVudGlvbiBoZWFkcykKICAgIC0gbl9wb3NpdGlvbnMgPSBtYXhfbGVuZ3RoIChDb250ZXh0IHdpbmRvdyBsZW5ndGgpCiAgICAtIHBhZF90b2tlbl9pZCA9IDAKIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbG9nZ2luZwpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSB0eXBpbmcgaW1wb3J0IERpY3QsIE9wdGlvbmFsLCBUdXBsZSwgVW5pb24KCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IEdQVDJDb25maWcsIEdQVDJMTUhlYWRNb2RlbAoKbG9nZ2luZy5iYXNpY0NvbmZpZygKICAgIGxldmVsPWxvZ2dpbmcuSU5GTywKICAgIGZvcm1hdD0iWyUoYXNjdGltZSlzXSBbJShsZXZlbG5hbWUpc10gKCUobmFtZSlzKSAlKG1lc3NhZ2UpcyIsCiAgICBkYXRlZm10PSIlWS0lbS0lZCAlSDolTTolUyIsCikKbG9nZ2VyID0gbG9nZ2luZy5nZXRMb2dnZXIoIk1vZGVsIikKCgpAZGF0YWNsYXNzCmNsYXNzIFJlY1N5c091dHB1dDoKICAgICIiIgogICAgU3RhbmRhcmRpemVkIG1vZGVsIG91dHB1dCBjb250YWluZXIgZm9yIHNlcXVlbnRpYWwgcmVjb21tZW5kYXRpb24uCiAgICAKICAgIEF0dHJpYnV0ZXM6CiAgICAgICAgbG9naXRzOiBGbG9hdFRlbnNvciBbQiwgTCwgdm9jYWJfc2l6ZV0gdW5ub3JtYWxpemVkIHByZWRpY3Rpb24gc2NvcmVzLgogICAgICAgIGxvc3M6IE9wdGlvbmFsIHNjYWxhciBjcm9zcy1lbnRyb3B5IGxvc3MgKGlmIGxhYmVscyB3ZXJlIHByb3ZpZGVkKS4KICAgICAgICBsYXN0X2hpZGRlbl9zdGF0ZTogT3B0aW9uYWwgRmxvYXRUZW5zb3IgW0IsIEwsIG5fZW1iZF0gY29udGV4dHVhbCByZXByZXNlbnRhdGlvbnMuCiAgICAiIiIKICAgIGxvZ2l0czogdG9yY2guVGVuc29yCiAgICBsb3NzOiBPcHRpb25hbFt0b3JjaC5UZW5zb3JdID0gTm9uZQogICAgbGFzdF9oaWRkZW5fc3RhdGU6IE9wdGlvbmFsW3RvcmNoLlRlbnNvcl0gPSBOb25lCgoKY2xhc3MgUmVjU3lzR1BUMihubi5Nb2R1bGUpOgogICAgIiIiCiAgICBTZXF1ZW5jZS1iYXNlZCBSZWNvbW1lbmRhdGlvbiBNb2RlbCB1c2luZyBhIENhdXNhbCBHUFQtMiBUcmFuc2Zvcm1lciBpbml0aWFsaXplZCBmcm9tIHNjcmF0Y2guCgogICAgSW4gTmV4dC1JdGVtIFByZWRpY3Rpb24sIGVhY2ggaW50ZXJhY3Rpb24gdG9rZW4gc2VxdWVuY2UgW3NfMSwgLi4uLCBzX2tdIGlzIHByb2plY3RlZAogICAgaW50byBhbiBlbWJlZGRpbmcgc3BhY2UsIHBhc3NlZCB0aHJvdWdoIGNhdXNhbCBzZWxmLWF0dGVudGlvbiBsYXllcnMgd2l0aCBhdXRvcmVncmVzc2l2ZQogICAgbWFza2luZywgYW5kIHByb2plY3RlZCB0byBjYXRhbG9nIGxvZ2l0cyBbQiwgTCwgdm9jYWJfc2l6ZV0uCgogICAgQXJnczoKICAgICAgICBudW1faXRlbXM6IFRvdGFsIG51bWJlciBvZiBhY3RpdmUgaXRlbXMgaW4gY2F0YWxvZyAoZXhjbHVkaW5nIDxQQUQ+KS4KICAgICAgICBtYXhfbGVuZ3RoOiBNYXhpbXVtIHNlcXVlbmNlIGNvbnRleHQgbGVuZ3RoIChuX3Bvc2l0aW9ucykuCiAgICAgICAgbl9lbWJkOiBMYXRlbnQgZW1iZWRkaW5nIGRpbWVuc2lvbiAoZGVmYXVsdDogMTI4KS4KICAgICAgICBuX2xheWVyOiBOdW1iZXIgb2YgdHJhbnNmb3JtZXIgZGVjb2RlciBsYXllcnMgKGRlZmF1bHQ6IDMpLgogICAgICAgIG5faGVhZDogTnVtYmVyIG9mIGF0dGVudGlvbiBoZWFkcyAoZGVmYXVsdDogNCkuCiAgICAgICAgZHJvcG91dDogRHJvcG91dCBwcm9iYWJpbGl0eSBhcHBsaWVkIHRvIGVtYmVkZGluZ3MgYW5kIHJlc2lkdWFsIGNvbm5lY3Rpb25zLgogICAgICAgIHBhZF90b2tlbl9pZDogUmVzZXJ2ZWQgcGFkZGluZyB0b2tlbiBpbmRleCAoZGVmYXVsdDogMCkuCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICBudW1faXRlbXM6IGludCwKICAgICAgICBtYXhfbGVuZ3RoOiBpbnQgPSA2NCwKICAgICAgICBuX2VtYmQ6IGludCA9IDEyOCwKICAgICAgICBuX2xheWVyOiBpbnQgPSAzLAogICAgICAgIG5faGVhZDogaW50ID0gNCwKICAgICAgICBkcm9wb3V0OiBmbG9hdCA9IDAuMSwKICAgICAgICBwYWRfdG9rZW5faWQ6IGludCA9IDAsCiAgICApIC0+IE5vbmU6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5udW1faXRlbXM6IGludCA9IG51bV9pdGVtcwogICAgICAgIHNlbGYubWF4X2xlbmd0aDogaW50ID0gbWF4X2xlbmd0aAogICAgICAgIHNlbGYucGFkX3Rva2VuX2lkOiBpbnQgPSBwYWRfdG9rZW5faWQKCiAgICAgICAgIyBWb2NhYiBzaXplID0gbnVtX2l0ZW1zICsgMSAoaXRlbSBJRHMgMS4uTiwgcGx1cyAwIGZvciBQQUQpCiAgICAgICAgdm9jYWJfc2l6ZSA9IG51bV9pdGVtcyArIDEKCiAgICAgICAgIyBDb25maWd1cmUgY3VzdG9tIEdQVC0yIGFyY2hpdGVjdHVyZQogICAgICAgIHNlbGYuY29uZmlnID0gR1BUMkNvbmZpZygKICAgICAgICAgICAgdm9jYWJfc2l6ZT12b2NhYl9zaXplLAogICAgICAgICAgICBuX3Bvc2l0aW9ucz1tYXhfbGVuZ3RoLAogICAgICAgICAgICBuX2N0eD1tYXhfbGVuZ3RoLAogICAgICAgICAgICBuX2VtYmQ9bl9lbWJkLAogICAgICAgICAgICBuX2xheWVyPW5fbGF5ZXIsCiAgICAgICAgICAgIG5faGVhZD1uX2hlYWQsCiAgICAgICAgICAgIHJlc2lkX3Bkcm9wPWRyb3BvdXQsCiAgICAgICAgICAgIGVtYmRfcGRyb3A9ZHJvcG91dCwKICAgICAgICAgICAgYXR0bl9wZHJvcD1kcm9wb3V0LAogICAgICAgICAgICBwYWRfdG9rZW5faWQ9cGFkX3Rva2VuX2lkLAogICAgICAgICAgICBib3NfdG9rZW5faWQ9Tm9uZSwgICMgQ2xlYXIgZGVmYXVsdCB0ZXh0IE5MUCB0b2tlbnMKICAgICAgICAgICAgZW9zX3Rva2VuX2lkPU5vbmUsCiAgICAgICAgKQoKICAgICAgICAjIFNUUklDVCBSRVFVSVJFTUVOVDogSW5pdGlhbGl6ZWQgZnJvbSBzY3JhdGNoIChOT1QgZnJvbSBwcmUtdHJhaW5lZCB3ZWlnaHRzKQogICAgICAgIHNlbGYudHJhbnNmb3JtZXIgPSBHUFQyTE1IZWFkTW9kZWwoc2VsZi5jb25maWcpCgogICAgICAgICMgQ3Jvc3MtRW50cm9weSBMb3NzIHdpdGggaWdub3JlX2luZGV4PTAgKDxQQUQ+KQogICAgICAgIHNlbGYubG9zc19mY3QgPSBubi5Dcm9zc0VudHJvcHlMb3NzKGlnbm9yZV9pbmRleD1zZWxmLnBhZF90b2tlbl9pZCkKCiAgICAgICAgIyBDb3VudCBwYXJhbWV0ZXJzCiAgICAgICAgbnVtX3BhcmFtcyA9IHN1bShwLm51bWVsKCkgZm9yIHAgaW4gc2VsZi5wYXJhbWV0ZXJzKCkpCiAgICAgICAgbnVtX3RyYWluYWJsZSA9IHN1bShwLm51bWVsKCkgZm9yIHAgaW4gc2VsZi5wYXJhbWV0ZXJzKCkgaWYgcC5yZXF1aXJlc19ncmFkKQogICAgICAgIGxvZ2dlci5pbmZvKAogICAgICAgICAgICAiSW5pdGlhbGl6ZWQgUmVjU3lzR1BUMiBmcm9tIHNjcmF0Y2g6IHZvY2FiX3NpemU9JWQgKG51bV9pdGVtcz0lZCArIFBBRCksICIKICAgICAgICAgICAgIm5fZW1iZD0lZCwgbl9sYXllcj0lZCwgbl9oZWFkPSVkLCBtYXhfbGVuZ3RoPSVkLiAiCiAgICAgICAgICAgICJUb3RhbCBwYXJhbWV0ZXJzOiAlZCAodHJhaW5hYmxlOiAlZCkiLAogICAgICAgICAgICB2b2NhYl9zaXplLAogICAgICAgICAgICBudW1faXRlbXMsCiAgICAgICAgICAgIG5fZW1iZCwKICAgICAgICAgICAgbl9sYXllciwKICAgICAgICAgICAgbl9oZWFkLAogICAgICAgICAgICBtYXhfbGVuZ3RoLAogICAgICAgICAgICBudW1fcGFyYW1zLAogICAgICAgICAgICBudW1fdHJhaW5hYmxlLAogICAgICAgICkKCiAgICBkZWYgZm9yd2FyZCgKICAgICAgICBzZWxmLAogICAgICAgIGlucHV0X2lkczogdG9yY2guVGVuc29yLAogICAgICAgIGF0dGVudGlvbl9tYXNrOiBPcHRpb25hbFt0b3JjaC5UZW5zb3JdID0gTm9uZSwKICAgICAgICBsYWJlbHM6IE9wdGlvbmFsW3RvcmNoLlRlbnNvcl0gPSBOb25lLAogICAgKSAtPiBSZWNTeXNPdXRwdXQ6CiAgICAgICAgIiIiCiAgICAgICAgRm9yd2FyZCBwYXNzIHRocm91Z2ggQ2F1c2FsIEdQVC0yLgoKICAgICAgICBBcmdzOgogICAgICAgICAgICBpbnB1dF9pZHM6IExvbmdUZW5zb3IgW0IsIExdIGNvbnRhaW5pbmcgaXRlbSB0b2tlbnMuCiAgICAgICAgICAgIGF0dGVudGlvbl9tYXNrOiBPcHRpb25hbCBMb25nVGVuc29yIFtCLCBMXSB3aXRoIDEgZm9yIHJlYWwgaXRlbXMsIDAgZm9yIDxQQUQ+LgogICAgICAgICAgICBsYWJlbHM6IE9wdGlvbmFsIExvbmdUZW5zb3IgW0IsIExdIGNvbnRhaW5pbmcgc2hpZnRlZCBncm91bmQtdHJ1dGggaXRlbSB0b2tlbnMuCgogICAgICAgIFJldHVybnM6CiAgICAgICAgICAgIFJlY1N5c091dHB1dCBjb250YWluaW5nIGxvZ2l0cyBbQiwgTCwgdm9jYWJfc2l6ZV0sIG9wdGlvbmFsIHNjYWxhciBsb3NzLAogICAgICAgICAgICBhbmQgb3B0aW9uYWwgbGFzdCBoaWRkZW4gc3RhdGUgcmVwcmVzZW50YXRpb25zIFtCLCBMLCBuX2VtYmRdLgogICAgICAgICIiIgogICAgICAgICMgQXV0by1nZW5lcmF0ZSBhdHRlbnRpb24gbWFzayBpZiBvbWl0dGVkCiAgICAgICAgaWYgYXR0ZW50aW9uX21hc2sgaXMgTm9uZToKICAgICAgICAgICAgYXR0ZW50aW9uX21hc2sgPSAoaW5wdXRfaWRzICE9IHNlbGYucGFkX3Rva2VuX2lkKS5sb25nKCkKCiAgICAgICAgIyBGb3J3YXJkIHRocm91Z2ggdHJhbnNmb3JtZXIgKHdpdGhvdXQgaW50ZXJuYWwgbGFiZWxzIHRvIGNvbXB1dGUgY3VzdG9tIGxvc3MpCiAgICAgICAgdHJhbnNmb3JtZXJfb3V0cHV0cyA9IHNlbGYudHJhbnNmb3JtZXIoCiAgICAgICAgICAgIGlucHV0X2lkcz1pbnB1dF9pZHMsCiAgICAgICAgICAgIGF0dGVudGlvbl9tYXNrPWF0dGVudGlvbl9tYXNrLAogICAgICAgICAgICBvdXRwdXRfaGlkZGVuX3N0YXRlcz1UcnVlLAogICAgICAgICAgICByZXR1cm5fZGljdD1UcnVlLAogICAgICAgICkKCiAgICAgICAgbG9naXRzID0gdHJhbnNmb3JtZXJfb3V0cHV0cy5sb2dpdHMgICMgW0IsIEwsIHZvY2FiX3NpemVdCiAgICAgICAgbGFzdF9oaWRkZW5fc3RhdGUgPSB0cmFuc2Zvcm1lcl9vdXRwdXRzLmhpZGRlbl9zdGF0ZXNbLTFdICAjIFtCLCBMLCBuX2VtYmRdCgogICAgICAgIGxvc3MgPSBOb25lCiAgICAgICAgaWYgbGFiZWxzIGlzIG5vdCBOb25lOgogICAgICAgICAgICAjIEZsYXR0ZW4gYmF0Y2ggYW5kIHNlcXVlbmNlIGRpbWVuc2lvbnMgZm9yIGNyb3NzLWVudHJvcHkKICAgICAgICAgICAgIyBQYWRkZWQgcG9zaXRpb25zIGluIGxhYmVscyAodmFsdWUgMCkgYXJlIGlnbm9yZWQgaW4gbG9zcyBjb21wdXRhdGlvbgogICAgICAgICAgICBsb3NzID0gc2VsZi5sb3NzX2ZjdCgKICAgICAgICAgICAgICAgIGxvZ2l0cy52aWV3KC0xLCBsb2dpdHMuc2l6ZSgtMSkpLAogICAgICAgICAgICAgICAgbGFiZWxzLnZpZXcoLTEpLAogICAgICAgICAgICApCgogICAgICAgIHJldHVybiBSZWNTeXNPdXRwdXQoCiAgICAgICAgICAgIGxvZ2l0cz1sb2dpdHMsCiAgICAgICAgICAgIGxvc3M9bG9zcywKICAgICAgICAgICAgbGFzdF9oaWRkZW5fc3RhdGU9bGFzdF9oaWRkZW5fc3RhdGUsCiAgICAgICAgKQoKICAgIGRlZiBnZXRfbGFzdF9hY3Rpb25fc3RhdGVzKAogICAgICAgIHNlbGYsCiAgICAgICAgaW5wdXRfaWRzOiB0b3JjaC5UZW5zb3IsCiAgICAgICAgYXR0ZW50aW9uX21hc2s6IE9wdGlvbmFsW3RvcmNoLlRlbnNvcl0gPSBOb25lLAogICAgKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgIiIiCiAgICAgICAgRXh0cmFjdHMgdGhlIGNvbnRleHR1YWwgcmVwcmVzZW50YXRpb24gdmVjdG9yIGhfdCBhdCB0aGUgbGFzdCB2YWxpZCBpbnRlcmFjdGlvbiB0b2tlbi4KICAgICAgICAKICAgICAgICBFc3NlbnRpYWwgZm9yIFBoYXNlIDIgKFJMIEZpbmUtVHVuaW5nKTogVGhpcyB2ZWN0b3Igc2VydmVzIGFzIHRoZSBNRFAgU3RhdGUgc190CiAgICAgICAgZm9yIHRoZSBSTCBwb2xpY3kgYWN0b3IvY3JpdGljIG5ldHdvcmtzLgoKICAgICAgICBBcmdzOgogICAgICAgICAgICBpbnB1dF9pZHM6IExvbmdUZW5zb3IgW0IsIExdCiAgICAgICAgICAgIGF0dGVudGlvbl9tYXNrOiBPcHRpb25hbCBMb25nVGVuc29yIFtCLCBMXQoKICAgICAgICBSZXR1cm5zOgogICAgICAgICAgICBGbG9hdFRlbnNvciBbQiwgbl9lbWJkXSBzdGF0ZSByZXByZXNlbnRhdGlvbnMuCiAgICAgICAgIiIiCiAgICAgICAgaWYgYXR0ZW50aW9uX21hc2sgaXMgTm9uZToKICAgICAgICAgICAgYXR0ZW50aW9uX21hc2sgPSAoaW5wdXRfaWRzICE9IHNlbGYucGFkX3Rva2VuX2lkKS5sb25nKCkKCiAgICAgICAgb3V0ID0gc2VsZi5mb3J3YXJkKGlucHV0X2lkcz1pbnB1dF9pZHMsIGF0dGVudGlvbl9tYXNrPWF0dGVudGlvbl9tYXNrKQogICAgICAgIGhpZGRlbiA9IG91dC5sYXN0X2hpZGRlbl9zdGF0ZSAgIyBbQiwgTCwgbl9lbWJkXQoKICAgICAgICAjIEZpbmQgdGhlIGluZGV4IG9mIHRoZSBsYXN0IGFjdGl2ZSB0b2tlbiBpbiBlYWNoIHNlcXVlbmNlCiAgICAgICAgIyBTZXF1ZW5jZSBsZW5ndGhzOiBzdW0gYWNyb3NzIGF0dGVudGlvbl9tYXNrCiAgICAgICAgc2VxX2xlbnMgPSBhdHRlbnRpb25fbWFzay5zdW0oZGltPS0xKSAtIDEgICMgMC1pbmRleGVkIFtCXQogICAgICAgIHNlcV9sZW5zID0gdG9yY2guY2xhbXAoc2VxX2xlbnMsIG1pbj0wKQoKICAgICAgICBiYXRjaF9pbmRpY2VzID0gdG9yY2guYXJhbmdlKGlucHV0X2lkcy5zaXplKDApLCBkZXZpY2U9aW5wdXRfaWRzLmRldmljZSkKICAgICAgICBsYXN0X3N0YXRlcyA9IGhpZGRlbltiYXRjaF9pbmRpY2VzLCBzZXFfbGVuc10gICMgW0IsIG5fZW1iZF0KICAgICAgICByZXR1cm4gbGFzdF9zdGF0ZXMKCiAgICBAdG9yY2gubm9fZ3JhZCgpCiAgICBkZWYgcHJlZGljdF9uZXh0X2l0ZW1zKAogICAgICAgIHNlbGYsCiAgICAgICAgaW5wdXRfaWRzOiB0b3JjaC5UZW5zb3IsCiAgICAgICAgYXR0ZW50aW9uX21hc2s6IE9wdGlvbmFsW3RvcmNoLlRlbnNvcl0gPSBOb25lLAogICAgICAgIHRvcF9rOiBpbnQgPSAxMCwKICAgICAgICBmaWx0ZXJfaGlzdG9yeTogYm9vbCA9IFRydWUsCiAgICApIC0+IFR1cGxlW3RvcmNoLlRlbnNvciwgdG9yY2guVGVuc29yXToKICAgICAgICAiIiIKICAgICAgICBHZW5lcmF0ZXMgVG9wLUsgbmV4dCBpdGVtIHJlY29tbWVuZGF0aW9ucyBmb3IgdGhlIGFjdGl2ZSB1c2VyIHNlcXVlbmNlLgoKICAgICAgICBBcmdzOgogICAgICAgICAgICBpbnB1dF9pZHM6IExvbmdUZW5zb3IgW0IsIExdCiAgICAgICAgICAgIGF0dGVudGlvbl9tYXNrOiBPcHRpb25hbCBMb25nVGVuc29yIFtCLCBMXQogICAgICAgICAgICB0b3BfazogTnVtYmVyIG9mIHJlY29tbWVuZGF0aW9ucyB0byByZXR1cm4gcGVyIHNlcXVlbmNlLgogICAgICAgICAgICBmaWx0ZXJfaGlzdG9yeTogSWYgVHJ1ZSwgbWFza3MgcHJldmlvdXNseSBpbnRlcmFjdGVkIGl0ZW1zIHNvIHRoZXkgYXJlIG5vdCByZS1yZWNvbW1lbmRlZC4KCiAgICAgICAgUmV0dXJuczoKICAgICAgICAgICAgVHVwbGUgb2YgKHRvcGtfc2NvcmVzIFtCLCB0b3Bfa10sIHRvcGtfaW5kaWNlcyBbQiwgdG9wX2tdKS4KICAgICAgICAiIiIKICAgICAgICBzZWxmLmV2YWwoKQogICAgICAgIGlmIGF0dGVudGlvbl9tYXNrIGlzIE5vbmU6CiAgICAgICAgICAgIGF0dGVudGlvbl9tYXNrID0gKGlucHV0X2lkcyAhPSBzZWxmLnBhZF90b2tlbl9pZCkubG9uZygpCgogICAgICAgIG91dCA9IHNlbGYuZm9yd2FyZChpbnB1dF9pZHM9aW5wdXRfaWRzLCBhdHRlbnRpb25fbWFzaz1hdHRlbnRpb25fbWFzaykKICAgICAgICBsb2dpdHMgPSBvdXQubG9naXRzICAjIFtCLCBMLCB2b2NhYl9zaXplXQoKICAgICAgICAjIEV4dHJhY3QgbG9naXRzIGNvcnJlc3BvbmRpbmcgdG8gdGhlIGxhc3Qgb2JzZXJ2ZWQgaXRlbQogICAgICAgIHNlcV9sZW5zID0gYXR0ZW50aW9uX21hc2suc3VtKGRpbT0tMSkgLSAxCiAgICAgICAgc2VxX2xlbnMgPSB0b3JjaC5jbGFtcChzZXFfbGVucywgbWluPTApCiAgICAgICAgYmF0Y2hfaW5kaWNlcyA9IHRvcmNoLmFyYW5nZShpbnB1dF9pZHMuc2l6ZSgwKSwgZGV2aWNlPWlucHV0X2lkcy5kZXZpY2UpCiAgICAgICAgbmV4dF9pdGVtX2xvZ2l0cyA9IGxvZ2l0c1tiYXRjaF9pbmRpY2VzLCBzZXFfbGVuc10gICMgW0IsIHZvY2FiX3NpemVdCgogICAgICAgICMgTmV2ZXIgcmVjb21tZW5kIHRoZSA8UEFEPiB0b2tlbiAoaW5kZXggMCkKICAgICAgICBuZXh0X2l0ZW1fbG9naXRzWzosIHNlbGYucGFkX3Rva2VuX2lkXSA9IC1mbG9hdCgiaW5mIikKCiAgICAgICAgaWYgZmlsdGVyX2hpc3Rvcnk6CiAgICAgICAgICAgICMgTWFzayBvdXQgaXRlbXMgdGhhdCBhcHBlYXIgaW4gdGhlIHVzZXIncyBoaXN0b3JpY2FsIGNvbnRleHQKICAgICAgICAgICAgaGlzdG9yeV9tYXNrID0gdG9yY2guemVyb3NfbGlrZShuZXh0X2l0ZW1fbG9naXRzLCBkdHlwZT10b3JjaC5ib29sKQogICAgICAgICAgICBoaXN0b3J5X21hc2suc2NhdHRlcl8oMSwgaW5wdXRfaWRzLCBUcnVlKQogICAgICAgICAgICBoaXN0b3J5X21hc2tbOiwgc2VsZi5wYWRfdG9rZW5faWRdID0gRmFsc2UgICMgcHJlc2VydmUgcGFkIGZsYWcKICAgICAgICAgICAgbmV4dF9pdGVtX2xvZ2l0cyA9IG5leHRfaXRlbV9sb2dpdHMubWFza2VkX2ZpbGwoaGlzdG9yeV9tYXNrLCAtZmxvYXQoImluZiIpKQoKICAgICAgICAjIENvbXB1dGUgdG9wX2sgc2NvcmVzIGFuZCBpbmRpY2VzCiAgICAgICAgdG9wa19zY29yZXMsIHRvcGtfaW5kaWNlcyA9IHRvcmNoLnRvcGsobmV4dF9pdGVtX2xvZ2l0cywgaz10b3BfaywgZGltPS0xKQogICAgICAgIHJldHVybiB0b3BrX3Njb3JlcywgdG9wa19pbmRpY2VzCgoKZGVmIGJ1aWxkX2dwdDJfcmVjc3lzX21vZGVsKAogICAgbnVtX2l0ZW1zOiBpbnQsCiAgICBtYXhfbGVuZ3RoOiBpbnQgPSA2NCwKICAgIG5fZW1iZDogaW50ID0gMTI4LAogICAgbl9sYXllcjogaW50ID0gMywKICAgIG5faGVhZDogaW50ID0gNCwKICAgIGRyb3BvdXQ6IGZsb2F0ID0gMC4xLAogICAgcGFkX3Rva2VuX2lkOiBpbnQgPSAwLAopIC0+IFJlY1N5c0dQVDI6CiAgICAiIiJGYWN0b3J5IGhlbHBlciB0byBidWlsZCBhbmQgcmV0dXJuIGEgUmVjU3lzR1BUMiBtb2RlbCBpbnN0YW5jZS4iIiIKICAgIHJldHVybiBSZWNTeXNHUFQyKAogICAgICAgIG51bV9pdGVtcz1udW1faXRlbXMsCiAgICAgICAgbWF4X2xlbmd0aD1tYXhfbGVuZ3RoLAogICAgICAgIG5fZW1iZD1uX2VtYmQsCiAgICAgICAgbl9sYXllcj1uX2xheWVyLAogICAgICAgIG5faGVhZD1uX2hlYWQsCiAgICAgICAgZHJvcG91dD1kcm9wb3V0LAogICAgICAgIHBhZF90b2tlbl9pZD1wYWRfdG9rZW5faWQsCiAgICApCg==",
    "train_base.py": "...",
    "requirements.txt": "dG9yY2g+PTIuMC4wCnRyYW5zZm9ybWVycz49NC4zNi4wCnBhbmRhcz49Mi4wLjAKbnVtcHk+PTEuMjIuMApweWFycm93Pj0xNC4wLjAKc2Npa2l0LWxlYXJuPj0xLjMuMAp0cWRtPj00LjY1LjAKd2FuZGI+PTAuMTYuMAo=",
}

for filename, encoded_content in EMBEDDED_MODULES.items():
    with open(filename, "wb") as f:
        f.write(base64.b64decode(encoded_content))
    print(f"[Workspace] Synced latest module: {filename}")

# 3. CRITICAL: Add current directory to Python sys.path so notebook cells can import data_utils, model, etc.
current_dir = os.path.abspath(os.getcwd())
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

print(f"\nActive working directory: {current_dir}")
print(f"Modules available: {[f for f in os.listdir(current_dir) if f.endswith(".py")]}")


## 3. Install Dependencies

In [ ]:
!pip install --quiet -r requirements.txt
print("Dependencies installed successfully!")

## 4. Optional: Weights & Biases (WandB) Setup
You can securely store your `WANDB_API_KEY` under Kaggle **Add-ons -> Secrets**.

In [ ]:
import os
USE_WANDB = False

try:
    from kaggle_secrets import UserSecretsClient
    import wandb
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret("WANDB_API_KEY")
    if wandb_key:
        wandb.login(key=wandb_key)
        print("[WandB] Successfully logged in via Kaggle Secrets!")
        USE_WANDB = True
    else:
        print("[WandB] WANDB_API_KEY secret is empty. Using local console logging.")
except Exception as e:
    print("[WandB] WandB secret not configured or offline mode. Running with local console logging.")

## 5. Dataset Setup: Download Official Yambda or Use Mounted Input\nThis cell checks for datasets mounted in `/kaggle/input/`. If none are mounted, it automatically downloads the official **Yambda-50M** interaction dataset (`flat/50m/listens.parquet`) from Hugging Face (**yandex/yambda**).

In [ ]:
# Set to True to download official Yambda-50M dataset from Hugging Face\nDOWNLOAD_REAL_YAMBDA = True\n\nimport os\nimport glob\nfrom huggingface_hub import hf_hub_download\n\nos.makedirs("/kaggle/working/data", exist_ok=True)\ndata_path = None\n\n# 1. Check if user attached a dataset via /kaggle/input/\ninput_files = glob.glob("/kaggle/input/**/*", recursive=True)\nparquet_files = [f for f in input_files if f.endswith(".parquet") or f.endswith(".csv")]\n\nif parquet_files:\n    data_path = parquet_files[0]\n    synthetic_flag = ""\n    print(f"[Dataset] Found mounted dataset in /kaggle/input/: {data_path}")\nelif DOWNLOAD_REAL_YAMBDA:\n    print("[Dataset] Downloading official Yambda-50M (yandex/yambda) from Hugging Face (~300MB)... ")\n    try:\n        data_path = hf_hub_download(\n            repo_id="yandex/yambda",\n            repo_type="dataset",\n            filename="flat/50m/listens.parquet",\n            local_dir="/kaggle/working/data"\n        )\n        synthetic_flag = ""\n        print(f"[Dataset] Successfully downloaded Yambda to: {data_path}")\n    except Exception as e:\n        print(f"[Dataset] Download failed ({e}). Falling back to synthetic generator.")\n        data_path = "/kaggle/working/data/synthetic_yambda.parquet"\n        synthetic_flag = "--synthetic"\nelse:\n    data_path = "/kaggle/working/data/synthetic_yambda.parquet"\n    synthetic_flag = "--synthetic"\n    print("[Dataset] Running in synthetic demo mode.")\n

## 6. Launch Training with GPU Acceleration & FP16
Executes `train_base.py` on the GPU with `--fp16`, `--split_mode global_temporal`, and evaluation on **HitRate@10** and **NDCG@10**.

In [ ]:
wandb_arg = "--use_wandb" if USE_WANDB else ""\n\n!python train_base.py \
    --data_path {data_path} \
    --processed_dir /kaggle/working/data/processed \
    --output_dir /kaggle/working/checkpoints \
    --user_col uid \
    --min_item_freq 5 \
    --max_length 64 \
    --n_embd 128 \
    --n_layer 3 \
    --n_head 4 \
    --batch_size 128 \
    --lr 5e-4 \
    --epochs 10 \
    --split_mode global_temporal \
    --fp16 \
    --num_workers 2 \
    {synthetic_flag} \
    {wandb_arg}

## 7. Model Verification & Top-K Recommendation Inference
Loads the best model checkpoint and tests next-item prediction on a user trajectory.

In [ ]:
import os
import sys
import torch

# Ensure current working directory is on sys.path
current_dir = os.path.abspath(os.getcwd())
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from data_utils import ItemTokenizer
from model import build_gpt2_recsys_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer_path = "/kaggle/working/data/processed/item_tokenizer.json"
ckpt_path = "/kaggle/working/checkpoints/best_gpt2_recsys.pt"

if os.path.exists(tokenizer_path) and os.path.exists(ckpt_path):
    tokenizer = ItemTokenizer.load(tokenizer_path)
    checkpoint = torch.load(ckpt_path, map_location=device)

    cfg = checkpoint["model_config"]
    model = build_gpt2_recsys_model(**cfg).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    print(f"Loaded Best Checkpoint (Epoch {checkpoint['epoch']}): Val HR@10 = {checkpoint['val_hr10']:.4f}, Val NDCG@10 = {checkpoint['val_ndcg10']:.4f}")

    # Demonstration: Sample next-item recommendation
    sample_input = torch.tensor([[1, 2, 3, 4]], dtype=torch.long, device=device)
    scores, topk = model.predict_next_items(sample_input, top_k=5)
    rec_items = tokenizer.inverse_transform(topk[0].cpu().numpy())
    print(f"\nInput History Tokens: {sample_input[0].tolist()}")
    print(f"Top-5 Recommended Next Tracks: {rec_items}")
else:
    print(f"Checkpoint or tokenizer not found. Make sure Cell 6 (training) has completed.")

## 8. Export Trained Artifacts
Compresses checkpoints and processed tokenizer mappings into a `.zip` archive for one-click download from `/kaggle/working`.

In [ ]:
!zip -r /kaggle/working/recsys_checkpoints.zip /kaggle/working/checkpoints /kaggle/working/data/processed
print("Checkpoints successfully archived to: /kaggle/working/recsys_checkpoints.zip")